# [Step 3] Physics-Informed Regression: 물리 기반 제약을 쓰는 아포지 예측
## Physics-Informed Regression for Rocket Apogee Prediction

> "인공지능은 데이터가 보여주는 진실을 물리 법칙이라는 경계선 안에서 찾는
>  제약 조건이 있는 최적화 도구이다."

기계공학에서 인공지능을 활용한다 함은, 

> "내가 설정한 특정 목적을 위해"
> "내가 보유하는 데이터를 적절히 처리하여"
> "가장 적절한 비선형 근사(추정)를 수행하는 과정"

이라고도 볼 수 있다.

## 0.1 UKF 의 성공과 한계

Step 2 에서 우리는 Unscented Kalman Filter(UKF) 를 통해 센서 노이즈를 줄인 상태 추정값 $(\hat{z}, \hat{v}_z)$ 을 얻었다. 아래 표는 **Bella Lui 예제 문서의 외부 사례**이며, 이 저장소의 200편 데이터(아포지 500~664 m, 최대 속도 43~78 m/s, 연소 종료 2.99 s)와는 다른 로켓이다.

(예시)
| 상태 | 시간 (s) | 고도 (m) | 속도 (m/s) | 운동에너지 (kJ) |
|------|---------|---------|-----------|---------------|
| 이륙 | 0.00 | 407 | 0 | 0 |
| 연소 종료 | 2.43 | 529.98 | 85.17 | 66.11 |
| 아포지 | 10.61 | 867.54 | 3.95 | ~0 |

EKF/UKF 의 성공 (Bella Lui 예제 기준):
- 연소 종료 시점 상태 추정 오차 < 5%
- 3-시그마 포함률 95% 이상

EKF/UKF 의 한계:
- 현재 구현은 상태 추정기이며, 미래 아포지 예측에는 별도의 forward model 또는 회귀 모델이 필요함.
- $C_d$ 조립 오차, 핀 각도 편차, 대기 밀도 변동 등 현재 모델에 명시하지 않은 불확실성은 별도 상태·파라미터 모델이나 보정 없이는 직접 반영하지 못함.
- 자이로 센서의 요동 (Spin), 기울기 (Tilt) 등 현재 EKF/UKF 구현은 이러한 특징과 아포지의 상관관계를 학습하지 않음.

> 현재 Step 3의 시뮬레이션 head에는 clean 고도·속도를 사용한다. Step 2의 추정값을 직접 연결한 평가나 실기 정확도 검증과는 구분한다.

## 0.2 AI 의 역할: 숨겨진 상관관계 학습

### 구체적 목표

1. 패턴 학습: 자이로, 틸트, 동압 등 다변수 관측 데이터와 에너지 손실 사이의 비선형 상관관계 포착
2. 물리 기반 제약: 입력 의존 bounded head로 coasting 출력의 상·하한을 제한 (전체 물리 법칙 보장 아님)
3. 외삽 일반화: 학습 범위 밖 성능은 아직 검증되지 않음
4. 운용 구간 선별: coasting이고 $v_z>3\,m/s$인 입력만 사용 (저속 발산 방지 보장 아님)

## 1.1 PINN 의 탄생 배경과 핵심 아이디어

### PINN 계보의 문제 인식 (Raissi et al., 2019; Karniadakis et al., 2021)

> "With the explosive growth of available data and computing resources, recent advances in machine learning and data analytics have yielded transformative results across diverse scientific disciplines. However, more often than not, in the course of analyzing complex physical, biological or engineering systems, the cost of data acquisition is prohibitive, and we are inevitably faced with the challenge of drawing conclusions and making decisions under partial information."

### 문제 인식

- 딥러닝은 이미지, NLP 등에서 혁명적 성과를 거둠.
- 그러나 과학/공학 문제에서는 데이터 획득 비용이 매우 비쌈.
- 적은 데이터(small data regime)에서도 작동하는 방법론이 필요함.

### 해결책

> "Let it be the principled physical laws that govern the time-dependent dynamics of a system, or some empirically validated rules or other domain expertise, this prior information can act as a regularization agent that constrains the space of admissible solutions to a manageable size."

### PINN 의 핵심 철학
1. 물리 법칙 = 정규화 (regularization) 도구
2. 데이터 + 물리 = 데이터 효율성 + 물리적 일관성
3. 블랙박스 AI → 물리적 구조와 제약 일부를 명시한 모델 (완전한 해석 가능성 보장 아님)

## 1.2 PINN 이 해결하는 두 가지 문제

### Problem 1: Data-driven Solution (순문제)

> "Given fixed model parameters $\lambda$, what can be said about the unknown hidden state $u(t,x)$ of the system?"

- 입력: 물리 파라미터 (예: 점성계수, 밀도) + 일부 관측 데이터
- 출력: 전체 시공간 영역에서의 해 $u(t,x)$
- 이 실습의 출력: 센서 스냅샷 → 아포지 보정 비율 r. 전체 궤적 복원이나 Cd·모터 파라미터 식별은 구현하지 않았다.

### Problem 2: Data-driven Discovery (역문제)

> "What are the parameters $\lambda$ that best describe the observed data?"

- 입력: 관측 데이터
- 출력: 물리 파라미터 (예: 항력계수 $C_d$)
- 이 실습 범위: 아포지 회귀. $C_d$·모터 성능 파라미터 식별은 구현되어 있지 않다.

## 1.2.1 근사 이론 (Universal Approximation Theorem)

### 정의 (Cybenko, 1989; Hornik, 1991)

> 하나의 은닉층을 가진 피드포워드 신경망은 유한한 수의 뉴런으로 임의의 연속 함수를 임의의 정밀도로 근사할 수 있다.

### 수식적 표현

$$ \forall f \in C(K), \forall \epsilon > 0, \exists N, W, b : \left| f(x) - \sum_{i=1}^{N} w_i \sigma(W_i x + b_i) \right| < \epsilon, \quad \forall x \in K $$

### 우리 문제에서의 의미
## 1.3 PINN 의 수학적 표현

### 일반적인 비선형 편미분방정식

$$u_t + \mathcal{N}[u; \lambda] = 0, \quad x \in \Omega, \quad t \in [0, T]$$

여기서:
- $u(t,x)$: 구하고자 하는 해 (예: 로켓의 고도, 속도)
- $\mathcal{N}[\cdot; \lambda]$: 비선형 미분 연산자 (물리 법칙)
- $\lambda$: 물리 파라미터 (역문제에서는 학습 대상)

### PINN 의 접근법

1. $u(t,x)$를 신경망으로 근사
2. Automatic Differentiation으로 미분항 계산
3. 물리 법칙을 loss function 에 포함

| 항목 | 칼만 필터 | 신경망 |
|------|----------|--------|
| 모델 형태 | 명시적 수식 ($\dot{v} = \frac{T-D}{m} - g$) | 암시적 함수 근사 ($f_\theta(x)$) |
| 비선형성 처리 | 자코비안/UT로 국소 근사 | 비선형 함수 근사 표현력 (전역 최적 보장 아님) |
| 불확실성 소스 | $Q, R$ 공분산, 상태 증강·적응 필터 | 별도 확률 모델·학습 목적·추론 방법 필요 |
| 한계 | 구조적 모델 오차는 $Q$로 근사 | 물리 법칙 보장 불가 |

### 결론

신경망은 비선형 보정 함수를 근사할 표현력을 가진다. 그러나 UAT는 compact 영역에서의 존재 정리이며,
학습 성공·외삽·불확실성 신뢰성을 보장하지 않는다. 칼만 필터의 모델 오차 표현과 신경망 잔차 학습은
서로 배타적이지 않다.

## 1.4 비보존력과 에너지 소모 (Aerodynamic Work)

### 전공지식: 보존력 vs 비보존력

| 힘 | 종류 | 퍼텐셜 에너지 정의 | 일 - 에너지 정리 |
|----|------|------------------|---------------|
| 중력 | 보존력 (Conservative) | $U_g = mgh$ | $\oint \vec{F}_g \cdot d\vec{s} = 0$ |
| 추력 | 외부 입력 | 정의 불가 | $W_T = \int \vec{T} \cdot d\vec{s}$ |
| 항력 | 비보존력 (Non-conservative) | 정의 불가 | $\oint \vec{F}_d \cdot d\vec{s} < 0$ |

### 에너지 소모 수식

연소 종료 후 (Coasting Phase) 에너지 방정식:

> 이 절의 에너지 소산식은 정지 공기·무추력·일정 질량을 가정한다. 아래 아포지 제약은 수직 비중력 일이 비양수인 상승 모델에 적용하며, h와 H는 같은 발사대 기준 AGL이다. 수직 등가 에너지는 $mgh+mv_z^2/2$이며 일반 3D 총에너지와 구분한다.

$$ E_{total}(t) = E_{kinetic}(t) + E_{potential}(t) $$

$$ \frac{dE_{total}}{dt} = \vec{F}_{drag} \cdot \vec{v} = -\frac{1}{2} \rho v^2 C_d A \cdot v = -\frac{1}{2} \rho C_d A v^3 $$

### 중요한 물리적 제약

$$ \frac{dE_{total}}{dt} = \vec{F}_{drag} \cdot \vec{v}_{ground} \leq 0 \quad \text{(정지 공기·무추력·항력만 가정)} $$

> 바람·양력·3D 운동이 있으면 상대속도와 각 힘의 일을 구분해야 한다. 이 식은 단순화된 모델의 조건부 결과다.

### 아포지 예측 식 (하한선 보장)

$$ h_{apogee} = (h_{curr} + \frac{v_z^2}{2g}) \cdot r, \quad \text{where } \frac{h_{curr}}{h_{curr}+v_z^2/(2g)} \le r \le 1 $$

$$ \text{고도 손실} = (h_{curr}+G) \cdot (1 - r) \quad (\text{관성 구간, } r \le 1) $$

> 에너지 손실 [J] 로 쓰려면 여기에 \(mg\) 를 곱해야 한다. \(G(1-r)\) 는 절대 아포지 배율 r 에 대해 성립하지 않는다.

PINN 이 학습할 대상:

$$ r = \frac{H}{h+G},\quad G=\frac{v_z^2}{2g},\quad \frac{h}{h+G}\le r\le 1 \quad \text{(수직 소산 상승 모델)} $$

$$ h_{curr}\le \hat{H}_{apogee}\le h_{curr}+G \quad \text{(다음 절의 bounded head로 보장)} $$

> 곱셈식만으로 하한이 보장되는 것은 아니다. 연소 구간에는 이 상한을 적용하지 않으며, 데이터의 작은 상한 초과도 clip하지 않고 감사한다. train/validation은 strict 범위 일관 행으로 학습·선택하고, 주 test는 제외 전의 유효·유한 라벨 후보 전체로 평가한다.

## 1.5 Hard vs Soft Constraint

우리는 인공지능이 "물리라는 선"을 넘지 않도록 hard와 soft라는 두 가지 제약 방식을 구분한다. 현재 실습은 hard head를 사용하며, 운용 안전성 보장과는 다르다.

### Hard Constraint: 구조적 정합성 강제

"모델의 형상 (Architecture) 자체가 물리 법칙을 포함하도록 설계하는 방식"

| 항목 | 설명 |
|------|------|
| 비유 | 물리적인 벽 (통과 불가) |
| 구현 위치 | 출력 레이어, 활성화 함수, Physics Layer |
| 적용 내용 | coasting 범위 ($h/(h+G)\le r\le1$), 현재 고도 하한과 탄도 상한 |
| 보장 범위 | 유한·유효 입력과 수치 오차 내에서 지정한 출력 제약을 구조적으로 만족. 예측 정확도·운용 안전성의 확률 보장은 아님 |

### 수정된 Hard Constraint 수식

$$ D=h_{curr}+\frac{v_z^2}{2g},\quad r_{floor}=\frac{h_{curr}}{D},\quad \sigma(a)=\frac{1}{1+e^{-a}} $$

$$ \hat{H}_{apogee} = D\cdot\underbrace{\left[r_{floor}+(1-r_{floor})\sigma(\text{ANN}(\text{inputs}))\right]}_{r_{floor}\le\hat r\le1} $$

### 물리적 보장

1.  $r \ge h_{curr}/(h_{curr}+v_z^2/2g)$ → $\hat{H}_{apogee} \ge h_{curr}$ 와 $r\le1$ → $\hat{H}_{apogee}\le h_{curr}+v_z^2/(2g)$ (유효 coasting 입력의 head에서 강제)
2.  연소 중에는 추력이 일을 하므로 $r>1$이 가능하지만, 이 모델은 연소 구간에 적용하지 않는다. 조건부 물리 상한과 실제 데이터의 작은 상한 초과는 구분하며 원인은 미확인이다.
3.  수학적으로 $v_z=0$, $h_{curr}>0$이면 $r_{floor}=1$이므로 $\hat{H}_{apogee}=h_{curr}$. 다만 현재 운용 범위는 $v_z>3\,m/s$이며 아포지 상태를 실행 검증한 것은 아니다.

> 보장 대상은 입력 상태에 대한 지정한 출력 제약(하한·탄도 상한)이다. 운동방정식 전체, 실제 상태의 정확성, 예측 정확도·안전성은 보장하지 않는다. 이전 $1+0.7\tanh$ 범위만으로는 r이 1을 넘어 탄도 상한을 위반할 수 있었지만, 현재 head는 이를 제한하며 사후 max clamp를 사용하지 않는다.

### Soft Constraint: Loss Function 을 통한 강제

"모델이 물리 법칙을 어길 수는 있지만, 어길 때마다 손실 (Loss) 을 높여 스스로 교정하게 하는 방식"

$$ \text{Total Loss}_{soft}=\underbrace{\mathcal{L}_{data}}_{\text{데이터 오차}}+\lambda\underbrace{\mathcal{L}_{physics}}_{\text{제약 위반 벌점}} $$

> 위 식은 soft 방식의 일반 개념이다. 현재 구현은 두 비교 모델에 동일한 지도 손실을 적용하고 range/coast/mono 벌점은 사용하지 않는다. 시간 좌표 편미분을 벌점하더라도 실제 궤적의 전미분 단조성이 보장되지는 않는다.

## 1.6 Inductive Bias: 인간 지식을 AI 에 주입하기

### 정의

> Inductive Bias 란 학습 알고리즘이 보지 못한 데이터에 대해 예측할 때 사용하는 사전 가정 (Prior Assumption)의 집합을 뜻함.

### 일반 ANN vs PINN 의 Inductive Bias 비교

| 항목 | 일반 ANN | PINN (우리 모델) |
|------|----------|-----------------|
| 사전 가정 | "데이터 패턴이 미래에도 지속될 것이다" | "수직 소산 가정의 출력 범위를 제한한다" |
| 외삽 성능 | 범위 위반 가능, 정확도 미보장 | 지정 출력 경계 준수, 정확도 미보장 |
| 데이터 효율 | 데이터와 문제에 의존 | 물리 제약의 효율 개선 효과는 비교 실험 필요 |
| 하한선 보장 | 없음 | 구조적 보장 |

### 요약

> 물리 지식으로 가설 공간을 제한할 수 있다. 그러나 더 적은 데이터로 더 좋은 성능을 내거나 OOD를 해결한다는 보장은 아니며, 독립 평가가 필요하다.

## 1.7 자동 미분 (Autograd) 과 시간축 미분

### Step 2(EKF) 와의 비교

| 항목 | Step 2 (EKF) | Step 3 (PINN) |
|------|-------------|--------------|
| 미분 대상 | 상태 전이 함수 $f(x)$ | 지도 손실과 bounded head의 가중치 경로 |
| 미분 방법 | 수동 자코비안 유도 | 자동 미분 (Autograd) |
| 시간 미분 | 이산 시간 상태 전파 | 현재 구현은 시간 좌표 미분을 사용하지 않음 |

### 핵심 전달 메시지

> "Autograd는 구현한 계산 그래프를 미분한다. 현재 실습은 물리 head를 통과해 가중치 gradient를 계산하며, 물리 법칙 자체를 발견하거나 시간 미분 제약을 계산하는 것은 아니다."

### 주의: 정규화 도메인 vs 물리 도메인

정규화된 도메인에서 계산한 미분값은 물리적 변화율과 다름:

$$ \frac{\partial \hat{r}}{\partial t_{normalized}} = \frac{\partial \hat{r}}{\partial t_{physical}} \cdot \sigma_t $$

시간 미분을 사용할 경우 위 chain rule로 물리 단위에 맞게 환산해야 한다. 이 관계식은 개념 설명이며, 현재 코드에는 시간 편미분 손실이 없다.

## 2.1 데이터 전처리의 철학: 정규화의 필요성과 물리적 의미 보존

우리가 이번 섹션에서 다루는 데이터 전처리 과정에는 두 가지 상반된 목표가 존재한다. 첫 번째는 모델의 학습 안정성을 위한 정규화 (Normalization), 두 번째는 물리 법칙을 적용하기 위한 물리적 단위 (Physical Units) 보존이다.

### 1. 신경망 학습의 안정성
심층 신경망은 입력 데이터의 스케일이 균일하지 않을 경우 그래디언트 소실 또는 폭발 문제를 겪기 쉽다. 예를 들어 고도 (m) 는 1000 단위이고, 속도 (m/s) 는 50 단위인 경우, 가중치 업데이트 비율이 달라져 학습이 불안정해질 수 있다. 따라서 StandardScaler 를 사용하여 모든 피처를 평균 0, 표준편차 1 로 맞춰주는 것은 학습 속도를 높이는 유용한 단계이다. 현재 scaler는 train에만 fit한다.

### 2. 물리 법칙을 위한 단위의 중요성
물리 head와 손실을 계산할 때 또한 주의가 필요하다. 물리 방정식은 단위를 포함하기 때문에 (kg, m/s 등), 만약 정규화된 입력 값으로만 미분을 수행하면, 계산되는 미분값이 실제 물리적 변화율을 망가뜨릴 수 있다. 
- 정규화 좌표에 대한 미분에는 해당 좌표의 scale factor가 들어간다. 에너지식·탄도식에는 정규화된 값을 물리량처럼 그대로 대입하지 않는다.

따라서 우리는 아래 두 가지 목적을 위해 데이터를 동시에 저장한다.
1. 정규화된 특징 배열 `x`: 신경망 입력으로 사용되어 학습 효율을 높임.
2. 물리 단위 상태 `h`, `v` 및 정답 `r`, `H`: head와 지도 손실·평가에서 물리적 의미를 유지함.

이러한 이중 전략은 학습용 scale과 물리 단위 계산을 분리한다. 현재 구현은 시간 미분용 별도 입력이 아니라, 정규화 특징과 원래 단위 상태를 보관한다.

In [ ]:
from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import apogee_pipeline as ap
import coasting_apogee as coast

SEED = 42
RUN_TRAINING = False  # 명시적으로 켜기 전에는 torch를 import하지 않습니다.
SAVE_ARTIFACTS = False
DATA_PATH = Path("data/simulated/all_trajectories.csv.gz")
META_PATH = Path("data/simulated/flight_metadata.csv.gz")
OUTPUT_DIR = Path("data/experiments/step3-coasting")
REAL_PATH = Path("data/112.txt")
REAL_LAUNCH_TIME_S = None  # 로그 시스템 시각을 외부 확인 후 설정합니다.
REAL_COAST_START_S = None  # 발사 후 초 단위입니다. 실기 연소 종료 확인 전에는 비워 둡니다.
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
H_SCALE_M = 100.0
H_LOSS_WEIGHT = 0.1
np.random.seed(SEED)
raw_df = pd.read_csv(DATA_PATH)
metadata_df = pd.read_csv(META_PATH)
print({"rows": len(raw_df), "flights": raw_df.flight_id.nunique(), "training_enabled": RUN_TRAINING})


In [ ]:
# 원본 energy_ratio를 덮어쓰지 않습니다. flight_phase가 없으면 명시적으로 실패합니다.
df = coast.make_coasting_frame(raw_df, metadata_df)
feature_cols = list(coast.FEATURE_COLUMNS)
if "energy_ratio" in raw_df:
    np.testing.assert_array_equal(df["energy_ratio"], raw_df["energy_ratio"])
print(df["input_rejection"].value_counts().to_string())
print(df["label_status"].value_counts().to_string())


In [ ]:
split = ap.build_split_manifest(df["flight_id"].unique(), seed=SEED)
parts, audit = coast.partition_coasting(df, split)
train_df = parts["train"]
val_df = parts["validation"]
cal_df = parts["calibration"]
test_df = parts["test"]
test_strict_df = parts["test_strict"]
print("비행 수:", {name: len(ids) for name, ids in split.items()})
print("행 감사: 입력 거부 이유는 상호 배타적이며 라벨 상태는 별도로 집계합니다.")
print(audit.to_string(index=False))
if train_df.empty or val_df.empty or test_df.empty:
    raise ap.PipelineError("train, validation and full test need nonempty valid populations")


## 2.4 PyTorch Dataset 클래스 구현 전략

Dataset/DataLoader에 넘길 때도 정규화 특징과 물리 상태를 구분해야 한다. 현재 구현은 `arrays_for`로 배열을 구성하고 `tensor_arrays`와 배치 인덱스로 학습하며, 별도의 Dataset 클래스와 시간 미분 필드는 사용하지 않는다.

### 데이터 저장 구조
| 피처 이름 | 용도 | 주석 |
|----------|------|------|
| `x` | 신경망 입력 | train-only StandardScaler를 적용한 특징 |
| `h`, `v` | 물리 head | AGL 고도와 수직 속도를 물리 단위로 유지 |
| `r`, `H` | 지도 손실 | 같은 AGL 기준 비율·아포지 정답 |
| `input_valid`, `label_in_bounds` | 구간·라벨 감사 | 입력 운용 조건과 정답 기반 strict 선별을 분리 |

### 왜 물리 단위가 필요한가?
일반적인 PINN에서 신경망 출력 $u(t,x)$의 좌표 편미분을 구하려면 실제 계산 경로의 좌표 입력이 `requires_grad=True`여야 한다. 현재 회귀에서는 좌표 미분을 계산하지 않는다. 이때 `time_normalized` 값을 사용하면 도함수 결과가 실제 물리적 크기를 반영하지 못한다. 반면 `time_physical` 를 사용하여 미분하면, 얻어지는 $\frac{\partial r}{\partial t}$ 는 실제 물리량 (1/초 단위) 로 해석될 수 있어 손실 함수의 스케일링 논리와 일관성을 유지할 수 있다.

현재 구현은 물리 단위의 h와 v로 범위를 구성한다. train/validation의 strict 라벨 선별과 전체 유효 test 평가는 구분하며, 시간 미분용 storage를 구현한 것으로 해석하지 않는다.

In [ ]:
feature_scaler = StandardScaler()
feature_scaler.fit(train_df[feature_cols])

def arrays_for(frame):
    x = (feature_scaler.transform(frame[feature_cols]).astype(np.float32)
         if len(frame) else np.empty((0, len(feature_cols)), dtype=np.float32))
    return {
        "x": x,
        "h": frame["altitude_rel"].to_numpy(dtype=np.float64),
        "v": frame["velocity_z"].to_numpy(dtype=np.float64),
        "r": frame["r_target"].to_numpy(dtype=np.float64),
        "H": frame["apogee_agl"].to_numpy(dtype=np.float64),
    }

arrays = {name: arrays_for(frame) for name, frame in parts.items()}
np.testing.assert_allclose(feature_scaler.mean_, train_df[feature_cols].mean().to_numpy())


In [ ]:
def build_models(input_dim, seed):
    # 실행 요청이 있을 때만 torch를 불러옵니다. 여기서는 소스만 제공합니다.
    import torch
    from torch import nn

    torch.manual_seed(seed)
    base = nn.Sequential(
        nn.Linear(input_dim, 64), nn.Tanh(),
        nn.Linear(64, 32), nn.Tanh(), nn.Linear(32, 1),
    )

    class BaselineANN(nn.Module):
        def __init__(self, network):
            super().__init__()
            self.network = network

        def forward(self, x, h, v):
            if (x.ndim != 2 or h.ndim != 1 or v.shape != h.shape
                    or len(x) != len(h)):
                raise ValueError("incompatible feature/state shapes")
            denom = h + v.square() / (2.0 * ap.SIM_G)
            valid = (torch.isfinite(x).all() & torch.isfinite(h).all()
                     & torch.isfinite(v).all() & torch.isfinite(denom).all()
                     & (h >= 0).all() & (v > 3).all() & (denom > 0).all())
            if not bool(valid):
                raise ValueError("requires finite coasting h >= 0, v > 3, D > 0")
            logit = self.network(x).squeeze(-1).to(h.dtype)
            if not bool(torch.isfinite(logit).all()):
                raise ValueError("nonfinite network output")
            return logit, denom * logit

    class ConstrainedANN(BaselineANN):
        def forward(self, x, h, v):
            logit, _ = super().forward(x, h, v)
            denom = h + v.square() / (2.0 * ap.SIM_G)
            r_floor = h / denom
            ratio = r_floor + (1.0 - r_floor) * torch.sigmoid(logit)
            return ratio, denom * ratio

    return {
        "baseline_ann": BaselineANN(copy.deepcopy(base)),
        "constrained_ann": ConstrainedANN(copy.deepcopy(base)),
    }


In [ ]:
def supervised_loss(r_pred, H_pred, r_target, H_target):
    # 두 모델에 동일한 무차원 지도 손실을 사용합니다.
    return ((r_pred - r_target).square().mean()
            + H_LOSS_WEIGHT * ((H_pred - H_target) / H_SCALE_M).square().mean())


In [ ]:
# 유효 영역, 역변환, 출력 경계의 NumPy 확인입니다. 학습 성능 시험은 아닙니다.
def check_numpy_contract(frame):
    restored = coast.reconstruct_apogee(frame["r_target"], frame["altitude_rel"], frame["velocity_z"])
    np.testing.assert_allclose(restored, frame["apogee_agl"], rtol=1e-12, atol=1e-10)
    for logit in (-1000.0, 0.0, 1000.0):
        ratio, height = coast.constrained_r_head(logit, frame["altitude_rel"], frame["velocity_z"])
        assert np.all(ratio <= 1.0)
        assert np.all(height >= frame["altitude_rel"].to_numpy() - 1e-10)
        assert np.all(height <= frame["denom"].to_numpy() + 1e-10)

check_numpy_contract(test_df)


In [ ]:
def tensor_arrays(data, device):
    import torch
    return {name: torch.as_tensor(value, device=device) for name, value in data.items()}


## 4.3 손실 함수 균형 잡기 (Loss Scale Balancing)

감독학습은 예측 오차를 최소화할 수 있고, soft-constraint PINN은 데이터 손실 ($\mathcal{L}_{data}$) 과 물리 법칙 위반 손실 ($\mathcal{L}_{physics}$) 을 함께 최소화한다. 여기서 중요한 문제는 두 손실 함수의 단위와 크기 차이가 매우 크다는 점이다.

### 단위 불일치 문제
- 데이터 손실 ($\mathcal{L}_{data}$): 에너지 비율 $r$의 제곱 오차이므로 무차원 단위 (크기 약 1e-3 ~ 1e-2 수준)
- 물리 손실 ($\mathcal{L}_{physics}$): 선택한 물리 항에 따라 단위와 크기가 다름. 시간 미분의 제곱 벌점이라면 $1/s^2$ 단위

만약 가중치 $\lambda$ 를 임의로 1.0 으로 설정한다면, 물리 손실이 데이터 손실보다 훨씬 커서 모델이 물리 법칙만 강제하고 실제 데이터를 무시하게 된다. 반대로 너무 작게 설정하면 물리 법칙을 지키지 않는 비현실적인 결과가 나올 수 있다.

### 해결 방법
학습 시작 전에 초기 배치 몇 개를 통해 두 손실의 크기를 측정하고, 서로 비슷한 크기가 되도록 $\lambda$ 값을 정하는 것은 **경험적 초기화** 방법이다. 비볼록 최적화의 안정적 수렴을 보장하지 않는다. 이전 학습 루프의 data/range/coast/mono 고정 가중치는 현재 사용하지 않는다. 현재 두 모델은 같은 무차원 지도 손실을 사용한다:

$$ \mathcal L=\operatorname{MSE}(\hat r,r)+0.1\,\operatorname{MSE}\!\left(\frac{\hat H-H}{100\,m},0\right) $$

상·하한은 head에서 강제하므로 range/coast/시간 편미분 벌점을 추가하지 않는다. 학습 성능은 아직 재실행하지 않았다.

In [ ]:
def train_one(model, train_data, validation_data):
    import torch
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device("cpu")
    model = model.to(device)
    train_t = tensor_arrays(train_data, device)
    val_t = tensor_arrays(validation_data, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []
    best_state = None
    best_val = float("inf")
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(train_t["x"]), device=device)
        loss_sum = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            r_pred, H_pred = model(train_t["x"][idx], train_t["h"][idx], train_t["v"][idx])
            loss = supervised_loss(r_pred, H_pred, train_t["r"][idx], train_t["H"][idx])
            if not bool(torch.isfinite(loss)):
                raise RuntimeError("nonfinite training loss")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach()) * len(idx)
        model.eval()
        with torch.no_grad():
            r_pred, H_pred = model(val_t["x"], val_t["h"], val_t["v"])
            validation_loss = float(supervised_loss(r_pred, H_pred, val_t["r"], val_t["H"]))
        if not np.isfinite(validation_loss):
            raise RuntimeError("nonfinite validation loss")
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / len(order),
                        "validation_loss": validation_loss})
        if validation_loss < best_val:
            best_val = validation_loss
            best_state = copy.deepcopy(model.state_dict())
    if best_state is None:
        raise RuntimeError("no finite best checkpoint")
    model.load_state_dict(best_state)
    model.eval()
    return model, history


## 4.4 자동 미분 구현 시 주의사항: 입력값에 대한 미분

PyTorch 에서 Automatic Differentiation 을 사용할 때 가장 흔한 실수는 `requires_grad` 속성을 잘못 설정하는 것이다. 일반적으로 가중치 (Weight) 에 대해서는 Backpropagation 을 통해 자동으로 미분 경로를 추적하지만, 입력 데이터 (Input Coordinate) 에 대해서는 명시적인 설정이 필요하다.

### 일반적인 오류 패턴
Dataset 클래스 내에서 `time_physical` 필드를 `requires_grad=True` 로 설정하여 데이터를 생성하려는 경우가 있다. DataLoader 사용 자체가 항상 gradient를 끊는 것은 아니다. 별도 복사한 시간이나 detach한 텐서를 미분 대상으로 삼으면 실제 출력의 계산 경로와 연결되지 않을 수 있다는 점을 확인해야 한다.

### 올바른 구현 방식
좌표 미분이 필요한 경우 실제 forward에 넣을 입력에 `requires_grad_(True)`를 설정하고 출력과의 연결을 확인한다. 독립 leaf가 필요할 때 `clone().detach()`를 쓸 수 있으나 모든 학습의 필수 절차는 아니다. 현재 Training Loop는 입력 시간 미분 없이 `loss.backward()`로 모델 가중치를 학습한다.

In [ ]:
models = {}
histories = {}
trained = False
if RUN_TRAINING:
    models = build_models(len(feature_cols), SEED)
    initial_states = [model.state_dict() for model in models.values()]
    for key in initial_states[0]:
        np.testing.assert_array_equal(initial_states[0][key].numpy(), initial_states[1][key].numpy())
    for name, model in models.items():
        models[name], histories[name] = train_one(model, arrays["train"], arrays["validation"])
    trained = True
else:
    print("skip: RUN_TRAINING=False; torch와 학습 모델을 실행하지 않았습니다.")


In [ ]:
def predict_ratios(model, data):
    import torch
    if len(data["x"]) == 0:
        return np.empty(0, dtype=float)
    device = next(model.parameters()).device
    values = tensor_arrays(data, device)
    predictions = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(values["x"]), BATCH_SIZE):
            stop = start + BATCH_SIZE
            ratio, _ = model(values["x"][start:stop], values["h"][start:stop], values["v"][start:stop])
            predictions.append(ratio.cpu().numpy())
    return np.concatenate(predictions)


In [ ]:
learned_metrics = {}
if trained:
    for name, model in models.items():
        learned_metrics[name] = {}
        for population in ("test", "test_strict"):
            ratio = predict_ratios(model, arrays[population])
            learned_metrics[name][population] = coast.score_predictions(parts[population], ratio)
    print("주 지표는 test이며 test_strict는 정답 경계가 맞는 보조 부분집합입니다.")
    print(json.dumps(learned_metrics, indent=2, ensure_ascii=False))
else:
    print("skip: 학습 모델 지표는 없습니다. 주 시험은 정답 경계로 선별하지 않습니다.")


In [ ]:
if trained:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, history in histories.items():
        table = pd.DataFrame(history)
        axes[0].plot(table["epoch"], table["train_loss"], label=name)
        axes[1].plot(table["epoch"], table["validation_loss"], label=name)
    axes[0].set_title("Training supervised loss")
    axes[1].set_title("Validation supervised loss")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.legend()
    fig.tight_layout()
    plt.show()


## 5.3 Stability Masking 구간의 성능 확인 (외삽 아님)

이전 구현에서는 학습 구간 ($v_z > 3.0 m/s$)과 별도로, **물리 손실을 끈** 저속 구간 ($v_z < 3.0 m/s$) 에서도 예측이 무너지지 않는지 확인했다. 당시 이 구간의 데이터는 데이터 손실에는 여전히 포함되어 있었으므로, 엄밀한 의미의 외삽(hold-out)이 아니라 **보간 성능 점검**이다.

### 왜 별도 점검이 필요한가
일반적인 딥러닝 모델은 훈련 데이터 범위 밖에서 예측 성능이 급격히 떨어진다. 이전 구현에서
Stability Masking은 **mono 항에만** 적용되고 range/coast 항은 전 구간에 적용된다. 따라서
저속 구간에서 물리 손실이 전부 꺼지는 것은 아니다. 진짜 외삽을 주장하려면 비행 단위로
저속 구간을 완전히 hold-out 해야 한다. 현재 버전은 $v_z>3\,m/s$의 상승 coasting 전용으로 저속 행을 학습·주 test 후보에서 제외하며, 저속 외삽 결과를 만들지 않는다.

### 검증 목표 (저속 운용을 별도로 확장할 경우)
1. 수렴성 확인: $v_z \to 0$ 일 때 예측 r이 1.0에 가까워지는가?
2. 안정성 확인: clamp 전/후와 r_floor 기준선을 분리해 비교하는가?

현재 다음 코드 셀은 저속 신경망 성능 대신 전체 유효 test와 strict 보조 subset에서 floor-only와 r=1 기준선을 별도로 보고한다. 위 저속 검증 목표를 실행한 것은 아니다.

In [ ]:
reference_metrics = {}
for population in ("test", "test_strict"):
    frame = parts[population]
    reference_metrics[population] = {
        "floor_only_not_learned": coast.score_predictions(frame, (frame["altitude_rel"] / frame["denom"]).to_numpy()),
        "constant_r_one_not_learned": coast.score_predictions(frame, np.ones(len(frame))),
    }
print("학습하지 않은 기준선입니다. ANN 성능은 아닙니다.")
print(json.dumps(reference_metrics, indent=2, ensure_ascii=False))


In [ ]:
# 기본값에서는 디렉터리도 만들지 않습니다. 기존 data/model 저장물은 사용하지 않습니다.
if SAVE_ARTIFACTS:
    if not trained:
        raise RuntimeError("refusing artifact write before explicit training")
    import joblib
    import torch
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, model in models.items():
        torch.save(model.state_dict(), OUTPUT_DIR / f"{name}_best.pt")
    joblib.dump(feature_scaler, OUTPUT_DIR / "feature_scaler.pkl")
    audit.to_csv(OUTPUT_DIR / "row_audit.csv", index=False)
    manifest = {
        "target": "r=H/(h+v_z^2/(2g)); H and h AGL", "features": feature_cols,
        "feature_units": ap.FEATURE_UNITS, "state_source": "simulation_clean",
        "split": split, "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "hidden_dims": [64, 32], "hidden_activation": "Tanh",
        "heads": {"baseline_ann": "linear r", "constrained_ann": "h/D+(1-h/D)*sigmoid(logit)"},
        "loss": {"r_mse_weight": 1.0, "H_mse_weight": H_LOSS_WEIGHT, "H_scale_m": H_SCALE_M},
        "data": str(DATA_PATH), "metadata": str(META_PATH),
        "data_sha256": ap.sha256_file(DATA_PATH), "metadata_sha256": ap.sha256_file(META_PATH),
        "primary_population": "all finite-label input-valid ascending coasting test rows",
        "secondary_population": "test rows satisfying h<=H<=D",
        "learned_metrics": learned_metrics, "nonlearned_reference_metrics": reference_metrics,
        "histories": histories,
    }
    ap.write_json(OUTPUT_DIR / "experiment.json", manifest)
else:
    print("skip: SAVE_ARTIFACTS=False; 새 산출물을 생성하지 않았습니다.")


In [ ]:
# 실기 연소 종료 시각을 시뮬레이션 설정에서 가져오지 않습니다.
real_df, real_status = coast.load_real_coasting(
    REAL_PATH, launch_time_s=REAL_LAUNCH_TIME_S, coast_start_s=REAL_COAST_START_S,
)
print(real_status)
# 독립 아포지 참조가 없으므로 실기 오차·MAE·RMSE를 만들지 않습니다.


In [ ]:
if real_df is not None and trained:
    import matplotlib.pyplot as plt
    real_data = {
        "x": feature_scaler.transform(real_df[feature_cols]).astype(np.float32),
        "h": real_df["altitude_rel"].to_numpy(dtype=np.float64),
        "v": real_df["velocity_z"].to_numpy(dtype=np.float64),
    }
    ratio = predict_ratios(models["constrained_ann"], real_data)
    prediction = coast.reconstruct_apogee(ratio, real_data["h"], real_data["v"])
    ballistic = real_data["h"] + real_data["v"]**2 / (2 * ap.SIM_G)
    fig, axis = plt.subplots(figsize=(10, 4))
    axis.plot(real_df["time"], real_data["h"], label="Current measured/filtered h AGL")
    axis.plot(real_df["time"], ballistic, label="Ballistic D (not learned)")
    axis.plot(real_df["time"], prediction, label="Constrained ANN forecast")
    axis.axhline(real_df.attrs["observed_max_agl"], linestyle=":",
                 label="Observed log maximum (not independent truth)")
    axis.set_xlabel("Time after configured launch (s)")
    axis.set_ylabel("AGL altitude (m)")
    axis.legend()
    fig.tight_layout()
    plt.show()
else:
    print("skip: 실기 관성 시작 설정과 새 학습 모델이 모두 있어야 진단 그림을 그립니다.")


## 7.1 ANN 은 패턴, PINN 은 물리 제약

> **이전 구현의 실행 기록 — 현재 모델 성과 아님:** 아래의 실기 “오차”는 독립 실제 아포지가 아니라 로그 최대값 대비 차이다. 로그는 아포지 전에 끊겼으므로 실제 아포지 MAE·RMSE로 해석하지 않는다.
>
> **실제 비행(112.txt) 결과 해석:** 최종 시점 clamp 후 오차는 +0.00 m지만, clamp 전 오차는
> **−3.52 m**, projected model −0.00 m, floor-only −0.00 m, clamp 활성 비율은 5.1%였다. 마지막 로그의
> v_z는 5.73 m/s로 영이 아니며, t=5.0 s에서는 −11.24 m (v_z=19.09 m/s)였다.
> 즉 +0.00 m는 학습 기여가 아니라 clamp가 현재 고도를 출력한 결과다. 참조 아포지도 독립 측정이
> 아니라 동일 kal 고도 로그의 최대값이다. 단일 비행 결과로 일반화하지 않는다.

### 7.1.1 물리 손실 ablation (이전 구현의 저장 지표)

아래 RMSE는 기존 저속 진단의 r 단위 지표다. 현재 coasting head·새 학습 결과가 아니며, 이전 clamp 효과를 해석하는 기록으로만 남긴다.

| 모델 | raw RMSE | clamp RMSE | clamp 비율 |
|:--|:--:|:--:|:--:|
| Baseline | 0.001916 | 0.000569 | 62.9% |
| PINN (coast off) | 0.140730 | 0.000359 | 100.0% |
| PINN (coast on) | 0.020638 | 0.000359 | 100.0% |
| r_floor only (신경망 없음) | — | **0.000359** | — |

동일 seed·절차에서 coast on은 raw RMSE를 0.1407 → 0.0206으로 줄였지만, clamp 후 두 PINN은
모두 r_floor 기준선(0.000359)과 같아진다. Baseline은 clamp 후 0.000569(활성 62.9%)다.
**저속 구간 헤드라인 지표는 학습이 아니라 clamp 구조가 지배한다.** PINN 우위/coast 효과는
raw 지표로만 해석해야 하며, raw에서도 PINN은 Baseline(0.001916)보다 나쁘다.

| 항목 | ANN | PINN |
|------|-----|------|
| 강점 | 데이터 패턴 학습 | 지정한 물리 출력 제약 준수 |
| 약점 | 범위 위반 가능 | 범위 안에서도 부정확할 수 있음 |
| 하한선 보장 | 없음 | 구조적 보장 |
| 구간 분리 | 같은 coasting 입력 | 같은 coasting 입력, strict train/validation |
| 아포지 안정성 | 운용 범위 밖 미검증 | 저속 운용·실기 정확도 미검증 |

## 7.2 현재 모델의 한계 (시계열 맥락 부재)

문제:
- 현재 모델은 "지금 이 순간" 의 데이터만 봄
- 과거 변화 이력은 현재 snapshot에서 알기 어려운 상태·기체 특성을 추정하는 데 도움이 될 수 있음 (기여량은 별도 비교 필요)

예시 (개념 설명이며 인과 검증 결과 아님):
- t=2s: 큰 기울기 발생 -> t=5s: 에너지 손실 증가 -> t=10s: 아포지 감소
- 현재 모델은 t=5s, t=5.2s 등 현재의 스냅샷만을 봄

> 현재 실습은 지도 회귀와 bounded head이며, PDE residual solver가 아니다. 새 모델의 학습·추론 성능은 미검증이다. 실기 연소 종료 시각 미확인 시 추론을 건너뛰고, 독립 아포지 정답이 없으면 정확도 지표를 만들지 않는다. Step 4 GRU는 별도 모델로 보존한다.

### 참고 문헌

1. Raissi, M., Perdikaris, P., & Karniadakis, G.E. (2019).
   Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. 
   *Journal of Computational Physics*, 378, 686-707.

2. Bella Lui - EPFL - 2020, RocketPy 1.11.0 documentation (5-6/15, 11-12/15, 14-15/15)

3. Cybenko, G. (1989). Approximation by Superpositions of a Sigmoidal Function.

4. Step 1 PDF

5. Step 1.5 & Step 2 Jupyter Notebook